In [ ]:
# pySCENIC pipeline (grn, ctx, aucell) on the timecourse object.
import scvelo as scv
import scanpy as sc
import cellrank as cr
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import palantir
import loompy as lp

In [ ]:
f_loom_path_scenic = "/home/yanjun2/scenic/option2_2.loom"
f_tfs = "/home/yanjun2/scenic/hs_hgnc_tfs.txt"

In [ ]:
!pyscenic grn {f_loom_path_scenic} {f_tfs} -o adj.csv --num_workers 10

In [ ]:
import glob
f_db_glob = "/home/yanjun2/scenic/hg38/*feather"
f_db_names = ' '.join( glob.glob(f_db_glob) )
print(f_db_names)

In [ ]:
!pyscenic ctx adj.csv \
    /home/yanjun2/scenic/hg38/hg38__refseq-r80__10kb_up_and_down_tss.mc9nr.genes_vs_motifs.rankings.feather /home/yanjun2/scenic/hg38/hg38__refseq-r80__500bp_up_and_100bp_down_tss.mc9nr.genes_vs_motifs.rankings.feather \
    --annotations_fname motifs-v9-nr.hgnc-m0.001-o0.0.tbl \
    --expression_mtx_fname {f_loom_path_scenic} \
    --output reg.csv \
    --mask_dropouts \
    --num_workers 12


In [ ]:
!pyscenic ctx adj.csv \
    /home/yanjun2/scenic/hg38/hg38__refseq-r80__10kb_up_and_down_tss.mc9nr.genes_vs_motifs.rankings.feather /home/yanjun2/scenic/hg38/hg38__refseq-r80__500bp_up_and_100bp_down_tss.mc9nr.genes_vs_motifs.rankings.feather \
    --annotations_fname motifs-v9-nr.hgnc-m0.001-o0.0.tbl \
    --expression_mtx_fname {f_loom_path_scenic} \
    --output reg.csv \
    --mask_dropouts \
    --num_workers 12

In [ ]:
f_pyscenic_output="/home/yanjun2/scenic/option2_2_SCENIC.loom"

In [ ]:
!pyscenic aucell \
    {f_loom_path_scenic} \
    reg.csv \
    --output {f_pyscenic_output} \
    --num_workers 10

In [ ]:
# Load data
import scanpy as sc
adata = sc.read_h5ad("/home/yanjun2/scQC/option2_2_final.h5ad")
adata

In [ ]:
!pip install cmake

In [ ]:
!pip install MulticoreTSNE

In [ ]:
import json
import zlib
import base64
import os
import numpy as np
import pandas as pd
import scanpy as sc
import loompy as lp
# collect SCENIC AUCell output
lf = lp.connect('option2_2_SCENIC.loom', mode='r+', validate=False )
auc_mtx = pd.DataFrame( lf.ca.RegulonsAUC, index=lf.ca.CellID)
lf.close()

In [ ]:
import umap

# UMAP
runUmap = umap.UMAP(n_neighbors=10, min_dist=0.4, metric='correlation').fit_transform
dr_umap = runUmap( auc_mtx )
pd.DataFrame(dr_umap, columns=['X', 'Y'], index=auc_mtx.index).to_csv( "scenic_umap.txt", sep='\t')
# tSNE
#tsne = TSNE( n_jobs=20 )
#dr_tsne = tsne.fit_transform( auc_mtx )
#pd.DataFrame(dr_tsne, columns=['X', 'Y'], index=auc_mtx.index).to_csv( "scenic_tsne.txt", sep='\t')

In [ ]:
#f_pyscenic_output="/home/yanjun2/scenic/option2_2_SCENIC.loom"

In [ ]:
# scenic output
lf = lp.connect( f_pyscenic_output, mode='r+', validate=False )
meta = json.loads(zlib.decompress(base64.b64decode( lf.attrs.MetaData )))
#exprMat = pd.DataFrame( lf[:,:], index=lf.ra.Gene, columns=lf.ca.CellID)
auc_mtx = pd.DataFrame( lf.ca.RegulonsAUC, index=lf.ca.CellID)
regulons = lf.ra.Regulons
dr_umap = pd.read_csv( 'scenic_umap.txt', sep='\t', header=0, index_col=0 )
#dr_tsne = pd.read_csv( 'scenic_tsne.txt', sep='\t', header=0, index_col=0 )
###



In [ ]:
auc_mtx.columns = auc_mtx.columns.str.replace('\(','_(')
regulons.dtype.names = tuple( [ x.replace("(","_(") for x in regulons.dtype.names ] )
# regulon thresholds
rt = meta['regulonThresholds']
for i,x in enumerate(rt):
    tmp = x.get('regulon').replace("(","_(")
    x.update( {'regulon': tmp} )



In [ ]:
# Neighbours, UMAP and clustering
import anndata as ad
ad_auc_mtx = ad.AnnData(auc_mtx)
sc.pp.neighbors(ad_auc_mtx, n_neighbors=10, metric="correlation")
sc.tl.umap(ad_auc_mtx)
sc.tl.tsne(ad_auc_mtx)

In [ ]:
adata.obsm["X_umap_aucell"] = ad_auc_mtx.obsm["X_umap"]
adata.obsm["X_tsne_aucell"] = ad_auc_mtx.obsm["X_tsne"]

In [ ]:
sc.pl.embedding(adata, basis="X_umap_aucell", color="condition")

In [ ]:
import seaborn as sns
auc_mtx["leiden_res_1_5_both_condition"] = adata.obs["leiden_res_1_5_both_condition"]
mean_auc_by_leiden_res_1_5_both_condition = auc_mtx.groupby("leiden_res_1_5_both_condition").mean()

In [ ]:
top_n = 500
top_tfs = mean_auc_by_leiden_res_1_5_both_condition.max(axis=0).sort_values(ascending=False).head(top_n)
mean_auc_by_leiden_res_1_5_both_condition_top_n = mean_auc_by_leiden_res_1_5_both_condition[
    [c for c in mean_auc_by_leiden_res_1_5_both_condition.columns if c in top_tfs]
]

In [ ]:
# Create the clustermap
g = sns.clustermap(
    mean_auc_by_leiden_res_1_5_both_condition_top_n,
    figsize=[200, 6.5],
    cmap="Blues",
    xticklabels=True,
    yticklabels=True
)

# Save the figure
g.savefig("/home/yanjun2/scenic/option22_clustermap.png", dpi=300)  # Adjust dpi as needed


In [ ]:
top_tfs

In [ ]:
tf_names = top_tfs.index.str.replace(r"\(\+\)", "", regex=True)
tf_names

In [ ]:
sc.pl.matrixplot(
    adata,
    tf_names,
    groupby="leiden_res_1_5_both_condition",
    cmap="Reds",
    dendrogram=False,
    figsize=[80, 5.5],
    standard_scale="group", save = 'option22scenic.png'
)

In [ ]:
#tsneDF = pd.DataFrame(adata.obsm['X_tsne'], columns=['_X', '_Y'])

Embeddings_X = pd.DataFrame(index=lf.ca.CellID)
Embeddings_X = pd.concat([
    pd.DataFrame(adata.obsm['X_umap'], index=adata.obs.index)[0],  # UMAP X
    pd.DataFrame(adata.obsm['X_pca'], index=adata.obs.index)[0]    # PCA X
], sort=False, axis=1, join='outer')
Embeddings_X.columns = ['1', '2']  # Adjust the number of columns as needed

# For Y embeddings, only include UMAP and PCA
Embeddings_Y = pd.DataFrame(index=lf.ca.CellID)
Embeddings_Y = pd.concat([
    pd.DataFrame(adata.obsm['X_umap'], index=adata.obs.index)[1],  # UMAP Y
    pd.DataFrame(adata.obsm['X_pca'], index=adata.obs.index)[1]    # PCA Y
], sort=False, axis=1, join='outer')
Embeddings_Y.columns = ['1', '2']  # Adjust the number of columns as needed


In [ ]:
adata

In [ ]:
import numpy as np
import pandas as pd

# Initialize metaJson
metaJson = {}

# Define embeddings, excluding t-SNE
metaJson['embeddings'] = [
    {
        "id": 1,
        "name": "Scanpy UMAP (highly variable genes)"
    },
    {
        "id": 2,
        "name": "Scanpy PC1/PC2"
    },
    {
        "id": 3,
        "name": "SCENIC AUC t-SNE"
    },
    {
        "id": 4,
        "name": "SCENIC AUC UMAP"
    },
]

# Define clusterings
metaJson["clusterings"] = [{
    "id": 0,
    "group": "Scanpy",
    "name": "Scanpy louvain 1.5 resolution",
    "clusters": [],
}]

# Define metrics
metaJson["metrics"] = [
    {"name": "nUMI"},
    {"name": "nGene"},
    {"name": "Percent_mito"}
]

# Define annotations based on the chosen metadata columns
metaJson["annotations"] = [
    {
        "name": "leiden_res_1_5_both_condition",
        "values": list(set(adata.obs['leiden_res_1_5_both_condition'].astype(str)))
    },
    {
        "name": "leiden_res_1_both_condition",
        "values": list(set(adata.obs['leiden_res_1_both_condition'].astype(str)))
    },
    {
        "name": "condition",
        "values": list(set(adata.obs['condition'].astype(str)))
    },
    {
        "name": "timepoint",
        "values": list(set(adata.obs['timepoint'].astype(str)))
    },
    {
        "name": "hash.ID",
        "values": list(set(adata.obs['hash.ID'].astype(str)))
    }
]

# SCENIC regulon thresholds (ensure rt is defined)
metaJson["regulonThresholds"] = rt

# Populate clusterings with cluster descriptions
for i in range(max(set([int(x) for x in adata.obs['leiden_res_1_5_both_condition']])) + 1):
    clustDict = {}
    clustDict['id'] = i
    clustDict['description'] = f'Unannotated Cluster {i + 1}'
    metaJson['clusterings'][0]['clusters'].append(clustDict)

# Create the clusterings DataFrame
clusterings = pd.DataFrame()
clusterings["0"] = adata.obs['leiden_res_1_5_both_condition'].values.astype(np.int64)



In [ ]:
def dfToNamedMatrix(df):
    arr_ip = [tuple(i) for i in df.values]
    dtyp = np.dtype(list(zip(df.dtypes.index, df.dtypes)))
    arr = np.array(arr_ip, dtype=dtyp)
    return arr


In [ ]:
adata

In [ ]:
col_attrs = {
    "CellID": np.array(adata.obs.index),
    "nUMI": np.array(adata.obs['nCount_RNA'].values),
    "nGene": np.array(adata.obs['nFeature_RNA'].values),
    "leiden_res_1_5_both_condition": np.array(adata.obs['leiden_res_1_5_both_condition'].values),
    "leiden_res_1_both_condition": np.array(adata.obs['leiden_res_1_both_condition'].values),
    "condition": np.array(adata.obs['condition'].values),
    "timepoint": np.array(adata.obs['timepoint'].values),
    "hash.ID": np.array(adata.obs['hash.ID'].values),
    "Percent_mito": np.array(adata.obs['percent.mt'].values),
    "Embeddings_X": dfToNamedMatrix(Embeddings_X),
    "Embeddings_Y": dfToNamedMatrix(Embeddings_Y),
    "RegulonsAUC": dfToNamedMatrix(auc_mtx),
    "Clusterings": dfToNamedMatrix(clusterings)
}

# Define row attributes
row_attrs = {
    "Gene": lf.ra.Gene,
    "Regulons": regulons,
}

# Define metadata attributes
attrs = {
    "title": "sampleTitle",
    "MetaData": json.dumps(metaJson),  # Ensure metaJson is correctly defined
    "Genome": 'hg38',
    "SCopeTreeL1": "",
    "SCopeTreeL2": "",
    "SCopeTreeL3": ""
}

# Compress the metadata field
attrs['MetaData'] = base64.b64encode(zlib.compress(json.dumps(metaJson).encode('ascii'))).decode('ascii')


In [ ]:
lp.create(
    filename = 'option2_2_scenic_final.loom' ,
    layers=lf[:,:],
    row_attrs=row_attrs, 
    col_attrs=col_attrs, 
    file_attrs=attrs
)
lf.close() # close original pyscenic loom file

In [ ]:
adata = sc.read( 'option2_2_scenic_final.loom', validate=False)

In [ ]:
lf = lp.connect('option2_2_scenic_final.loom', mode='r', validate=False )
meta = json.loads(zlib.decompress(base64.b64decode( lf.attrs.MetaData )))
exprMat = pd.DataFrame( lf[:,:], index=lf.ra.Gene, columns=lf.ca.CellID).T
auc_mtx = pd.DataFrame( lf.ca.RegulonsAUC, index=lf.ca.CellID)

In [ ]:
regulons = {}
for i,r in pd.DataFrame(lf.ra.Regulons,index=lf.ra.Gene).items():
    regulons[i] =  list(r[r==1].index.values)


In [ ]:
cellAnnot = pd.concat(
    [
        pd.DataFrame( lf.ca.leiden_res_1_5_both_condition, index=lf.ca.CellID ),
        pd.DataFrame( lf.ca.leiden_res_1_both_condition, index=lf.ca.CellID ),
        pd.DataFrame( lf.ca.condition, index=lf.ca.CellID ),
        pd.DataFrame( lf.ca.Percent_mito, index=lf.ca.CellID ),
        pd.DataFrame( lf.ca.nGene, index=lf.ca.CellID ),
        pd.DataFrame( lf.ca.nUMI, index=lf.ca.CellID ),
        pd.DataFrame( lf.ca.timepoint, index=lf.ca.CellID )
       
    ],
    axis=1
)
cellAnnot.columns = [
 'leiden_res_1_5_both_condition',
 'leiden_res_1_both_condition',
 'condition',
 'Percent_mito',
 'nGene',
 'nUMI',
  'timepoint']

In [ ]:
import pandas as pd

# Initialize list of DataFrames and their corresponding names
dr = []
dr_names = []

# Check if Embeddings_X and Embeddings_Y are available
if hasattr(lf.ca, 'Embeddings_X') and hasattr(lf.ca, 'Embeddings_Y'):
    drx = pd.DataFrame(lf.ca.Embeddings_X, index=lf.ca.CellID)
    dry = pd.DataFrame(lf.ca.Embeddings_Y, index=lf.ca.CellID)

    for i in range(len(drx.columns)):
        # Concatenate X and Y components for each embedding
        combined = pd.concat([drx.iloc[:, i], dry.iloc[:, i]], axis=1, join='outer')
        combined.columns = ['X', 'Y']  # Rename columns to 'X' and 'Y'
        dr.append(combined)
        
        # Append names to list, replacing spaces and slashes
        dr_names.append(meta['embeddings'][i]['name'].replace(" ", "_").replace('/', '-'))

# Check the DataFrames
for df in dr:
    print(df.head())


In [ ]:
from pyscenic.rss import regulon_specificity_scores
from pyscenic.plotting import plot_rss
import matplotlib.pyplot as plt
from adjustText import adjust_text
import seaborn as sns

In [ ]:
rss_timepoint = regulon_specificity_scores( auc_mtx, cellAnnot['timepoint'] )
rss_timepoint
rss_timepoint.to_csv('rss_timepoint.csv', index=True)

In [ ]:
rss_condition = regulon_specificity_scores( auc_mtx, cellAnnot['condition'] )
rss_condition

In [ ]:
rss_cellType = regulon_specificity_scores( auc_mtx, cellAnnot['leiden_res_1_5_both_condition'] )
rss_cellType


In [ ]:
import pandas as pd

# Assuming rss_cellType is a DataFrame
rss_cellType = regulon_specificity_scores(auc_mtx, cellAnnot['leiden_res_1_5_both_condition'])

# Save the DataFrame to a CSV file
rss_cellType.to_csv('rss_cellType.csv', index=True)  # Set index=True to include row indices in the CSV


In [ ]:
import matplotlib as mpl
def palplot(pal, names, colors=None, size=1):
    n = len(pal)
    f, ax = plt.subplots(1, 1, figsize=(n * size, size))
    ax.imshow(np.arange(n).reshape(1, n),
              cmap=mpl.colors.ListedColormap(list(pal)),
              interpolation="nearest", aspect="auto")
    ax.set_xticks(np.arange(n) - .5)
    ax.set_yticks([-.5, .5])
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    colors = n * ['k'] if colors is None else colors
    for idx, (name, color) in enumerate(zip(names, colors)):
        ax.text(0.0+idx, 0.0, name, color=color, horizontalalignment='center', verticalalignment='center')
    return f


In [ ]:
cats = sorted(list(set(cellAnnot['leiden_res_1_5_both_condition'])))
colors = sns.color_palette('bright',n_colors=len(cats) )
colorsd = dict( zip( cats, colors ))
colormap = [ colorsd[x] for x in cellAnnot['leiden_res_1_5_both_condition'] ]



In [ ]:
topreg = []
for i,c in enumerate(cats):
    topreg.extend(
        list(rss_cellType.T[c].sort_values(ascending=False)[:10].index)
    )
topreg = list(set(topreg))

In [ ]:
auc_mtx

In [ ]:
sorted_auc_mtx = auc_mtx.sort_values('leiden_res_1_5_both_condition')

In [ ]:
# Ensure 'leiden_res_1_5_both_condition' is not a Categorical type
sorted_auc_mtx = auc_mtx.sort_values('leiden_res_1_5_both_condition')

# Create the Z-scored version of the matrix
sorted_auc_mtx_Z = pd.DataFrame(index=sorted_auc_mtx.index)
for col in sorted_auc_mtx.columns:
    if pd.api.types.is_numeric_dtype(sorted_auc_mtx[col]):
        sorted_auc_mtx_Z[col] = (sorted_auc_mtx[col] - sorted_auc_mtx[col].mean()) / sorted_auc_mtx[col].std(ddof=0)
    else:
        # Optionally, handle non-numeric columns here
        sorted_auc_mtx_Z[col] = sorted_auc_mtx[col]  # Or set to NaN, if preferre


In [ ]:
# Create the clustermap with the sorted data
g = sns.clustermap(sorted_auc_mtx_Z[topreg], annot=False, square=False, linecolor='gray',
                   yticklabels=False, xticklabels=True, vmin=-2, vmax=6, 
                   row_colors=colormap, cmap="YlGnBu", figsize=(21,16))
g.cax.set_visible(True)
g.ax_heatmap.set_ylabel('')
g.ax_row_dendrogram.set_visible(False)
g.ax_col_dendrogram.set_visible(False)
g.ax_heatmap.set_xlabel('')


In [ ]:
auc_mtx_Z = pd.DataFrame( index=auc_mtx.index )
for col in list(auc_mtx.columns):
    auc_mtx_Z[ col ] = ( auc_mtx[col] - auc_mtx[col].mean()) / auc_mtx[col].std(ddof=0)

In [ ]:
sns.set()
sns.set(font_scale=0.8)
fig = palplot( colors, cats, size=1.0)
plt.savefig("anno_timepoint.pdf", dpi=600, bbox_inches = "tight")

In [ ]:
sns.set(font_scale=1.2)
g = sns.clustermap(auc_mtx_Z[topreg], annot=False,  square=False,  linecolor='gray',
    yticklabels=False, xticklabels=True, vmin=-2, vmax=6, row_colors=colormap,
    cmap="YlGnBu", figsize=(21,16) )
g.cax.set_visible(True)
g.ax_heatmap.set_ylabel('')
g.ax_row_dendrogram.set_visible(False)
g.ax_col_dendrogram.set_visible(False)
g.ax_heatmap.set_xlabel('')
#plt.savefig("timepoint-heatmap-top5.png", dpi=600, bbox_inches = "tight")



In [ ]:
sorted_auc_mtx_Z[topreg]

In [ ]:
# Sort the data based on the 'timepoint' column in cellAnnot
sorted_idx = cellAnnot.sort_values('timepoint').index
sorted_auc_mtx_Z = auc_mtx_Z.loc[sorted_idx]

# Recreate the colormap based on the sorted 'timepoint'
sorted_cellAnnot = cellAnnot.loc[sorted_idx]
colormap = [colorsd[x] for x in sorted_cellAnnot['timepoint']]

# Create the clustermap with the sorted data
g = sns.clustermap(sorted_auc_mtx_Z[topreg], annot=False, square=False, linecolor='gray',
                   yticklabels=False, xticklabels=True, vmin=-2, vmax=6, 
                   row_colors=colormap, cmap="YlGnBu", figsize=(21,16))
g.cax.set_visible(True)
g.ax_heatmap.set_ylabel('')
g.ax_row_dendrogram.set_visible(False)
g.ax_col_dendrogram.set_visible(False)
g.ax_heatmap.set_xlabel('')

# Save the plot if needed
# plt.savefig("timepoint-heatmap-top5.png", dpi=600, bbox_inches="tight")


In [ ]:
import warnings

warnings.filterwarnings("ignore")

import pyscenic
import loompy as lp
import scanpy as sc
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
import os

In [ ]:
ad_auc_mtx = ad.AnnData(auc_mtx)
sc.pp.neighbors(ad_auc_mtx, n_neighbors=10, metric="correlation")
sc.tl.umap(ad_auc_mtx)
sc.tl.tsne(ad_auc_mtx)